# Lorenz1stOrder

This notebook reproduces the tested Python/JAX workflow corresponding to `SSMTool/examples/Lorenz1stOrder/demo.mlx`: source model setup, linear eigenvalues, fixed-choice unstable SSM graph computation, linear reduced dynamics, reduced-to-full lifting, full Lorenz trajectory simulation, and the 3D SSM/full trajectory visualization.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt

from pathlib import Path
import sys

cwd = Path.cwd()
example_dir = cwd if (cwd / "lorenz.py").exists() else (cwd / "examples/lorenz_1st_order").resolve()
project_root = example_dir.parents[1]
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(example_dir))

from ssmtoolpy.core.graph import evaluate_univariate_graph, linear_reduced_trajectory, two_sided_graph_curve
from lorenz import lorenz_full_unstable_trajectories, lorenz_linear_eigenvalues, lorenz_rk4_trajectory, lorenz_ssm_invariance_residual, lorenz_unstable_ssm_graph_coefficients, lorenz_vector_field, standard_lorenz_parameters

sigma, rho, beta = standard_lorenz_parameters()
state = jnp.array([1.0, 2.0, 3.0])
lorenz_vector_field(state, sigma, rho, beta)

In [ ]:
jnp.sort(jnp.real(lorenz_linear_eigenvalues(sigma, rho, beta)))

In [ ]:
eigenvalue, eigenvector, coefficients = lorenz_unstable_ssm_graph_coefficients(sigma, rho, beta, order=5)
reduced = jnp.linspace(-1e-4, 1e-4, 5)
ssm_states = evaluate_univariate_graph(reduced, coefficients)
residual = lorenz_ssm_invariance_residual(reduced, eigenvalue, coefficients, sigma, rho, beta)
jnp.max(jnp.abs(residual))

In [ ]:
ssm_states

In [ ]:
times = jnp.linspace(0.0, 1.0, 501)
reduced_positive = linear_reduced_trajectory(1e-4, times, eigenvalue)
lifted_positive = evaluate_univariate_graph(reduced_positive, coefficients)
ssm_curve = two_sided_graph_curve(times, 1e-4, eigenvalue, coefficients)
full_trajectories = lorenz_full_unstable_trajectories(times, 1e-4, eigenvector, sigma, rho, beta)
lifted_positive[-1]

In [ ]:
comparison_times = jnp.linspace(0.0, 0.05, 101)
comparison_reduced = linear_reduced_trajectory(1e-5, comparison_times, eigenvalue)
comparison_lifted = evaluate_univariate_graph(comparison_reduced, coefficients)
comparison_full = lorenz_rk4_trajectory(comparison_lifted[0], comparison_times, sigma, rho, beta)
jnp.max(jnp.abs(comparison_full - comparison_lifted))

In [ ]:
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(projection="3d")
ax.plot(ssm_curve[:, 0], ssm_curve[:, 1], ssm_curve[:, 2], "b-", linewidth=1.5, label="SSM")
ax.plot(full_trajectories[:, 0], full_trajectories[:, 1], full_trajectories[:, 2], "ro", markersize=2.5, label="Full")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=35, azim=15)
ax.legend()
fig.tight_layout()